# Verification of Generated Texts — DeepSeek API — batch_07

Runs `prompt_verification.md` on every JSON file in `prompts_batches_20/batch_07/`.

- **System prompt**: `prompt_verification.md` (the auditor instructions)
- **User message**: the JSON object `{id, specie, specie_en, deb_model, prompt, generated_text}`
- **Result saved**: key `verification` added to each JSON file (skip if already present or if `generated_text` is missing)

In [1]:
import json
import os
import time
import urllib.request
import urllib.error
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path("__file__").resolve().parents[2] / ".env") if "__file__" in dir() else load_dotenv()

# ── Configuration ─────────────────────────────────────────────────────────────
API_KEY            = os.environ.get("DEEPSEEK_API_KEY")
MODEL              = "deepseek-v4-pro"
API_URL            = "https://api.deepseek.com/chat/completions"
GENERATED_DIR      = Path("../prompts_batches_20/batch_07")
PROMPT_FILE        = Path("../verif_prompts/prompt_verification.md")

DELAY_BETWEEN_CALLS = 5
MAX_RETRIES         = 6
RETRY_DELAY         = 30
# ──────────────────────────────────────────────────────────────────────────────

if not API_KEY:
    raise ValueError("DEEPSEEK_API_KEY environment variable is not set.")

SYSTEM_PROMPT = PROMPT_FILE.read_text(encoding="utf-8")

json_files = sorted(GENERATED_DIR.glob("*.json"))
print(f"Model          : {MODEL}")
print(f"API URL        : {API_URL}")
print(f"Found          : {len(json_files)} file(s) in {GENERATED_DIR}")


Model          : deepseek-v4-pro
API URL        : https://api.deepseek.com/chat/completions
Found          : 370 file(s) in ../prompts_batches_20/batch_07


In [2]:
def call_deepseek_verify(system_prompt: str, user_content: str) -> tuple[str, dict]:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_content},
    ]
    body = json.dumps({"model": MODEL, "messages": messages}).encode()
    req = urllib.request.Request(
        API_URL,
        data=body,
        headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=180) as resp:
        data = json.loads(resp.read().decode())
    return data["choices"][0]["message"]["content"], data.get("usage", {})


# ── Main loop ─────────────────────────────────────────────────────────────────
fatal = False
for idx, filepath in enumerate(json_files, 1):
    data = json.loads(filepath.read_text(encoding="utf-8"))

    if data.get("verification") is not None:
        print(f"[SKIP-verified]  {filepath.name}")
        continue

    if not data.get("generated_text"):
        print(f"[SKIP-no-text]   {filepath.name}")
        continue

    # Build the user message: only the fields the auditor needs
    user_payload = json.dumps({
        "id":             data.get("id"),
        "specie":         data.get("specie"),
        "specie_en":      data.get("specie_en"),
        "deb_model":      data.get("deb_model"),
        "prompt":         data.get("prompt"),
        "generated_text": data.get("generated_text"),
    }, ensure_ascii=False)

    print(f"[{idx:04d}/{len(json_files)}] {filepath.name} ...", end=" ", flush=True)

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            raw_result, usage = call_deepseek_verify(SYSTEM_PROMPT, user_payload)

            # Parse the JSON response from DeepSeek
            try:
                verification = json.loads(raw_result)
            except json.JSONDecodeError:
                # Sometimes the model wraps the JSON in ```json ... ```
                stripped = raw_result.strip().removeprefix("```json").removesuffix("```").strip()
                verification = json.loads(stripped)

            data["verification"] = verification
            filepath.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

            verdict = verification.get("verdict", "?")
            critical = verification.get("critical_errors", "?")
            cached = usage.get("prompt_cache_hit_tokens", 0)
            print(f"{verdict} | {critical} critical | {usage.get('prompt_tokens','?')} in / {usage.get('completion_tokens','?')} out / {cached} cached")
            break

        except urllib.error.HTTPError as e:
            body_err = e.read().decode(errors="replace")
            print(f"\n  HTTP {e.code} on attempt {attempt}: {body_err[:200]}")
            if e.code == 401:
                print("  Invalid API key — check DEEPSEEK_API_KEY. Aborting.")
                fatal = True
                break
            elif e.code == 404:
                print(f"  Model '{MODEL}' not found. Aborting.")
                fatal = True
                break
            elif e.code == 403:
                print("  Access forbidden — skipping this file.")
                break
            elif e.code == 429:
                wait = RETRY_DELAY * (2 ** (attempt - 1))
                print(f"  Rate limited — waiting {wait}s before retry...", flush=True)
                time.sleep(wait)
            elif attempt < MAX_RETRIES:
                time.sleep(5 * attempt)
        except Exception as e:
            print(f"\n  Error on attempt {attempt}: {e}")
            if attempt < MAX_RETRIES:
                time.sleep(5 * attempt)
    else:
        print(f"  FAILED after {MAX_RETRIES} attempts — skipping.")

    if fatal:
        break

    if idx < len(json_files):
        time.sleep(DELAY_BETWEEN_CALLS)

print("\nAll done.")

[0001/370] 2221_Elaenia_albiceps_std.json ... FAIL | 3 critical | 35847 in / 1730 out / 12032 cached
[0002/370] 2222_Elaenia_gigas_std.json ... FAIL | 5 critical | 35640 in / 2169 out / 12032 cached
[0003/370] 2223_Elaenia_obscura_std.json ... FAIL | 2 critical | 35837 in / 2016 out / 12032 cached
[0004/370] 2224_Elaenia_pallatangae_std.json ... FAIL | 2 critical | 35652 in / 1758 out / 12032 cached
[0005/370] 2225_Elagatis_bipinnulata_abj.json ... FAIL | 3 critical | 34334 in / 1922 out / 12032 cached
[0006/370] 2226_Elanus_caeruleus_std.json ... FAIL | 2 critical | 35003 in / 1627 out / 12032 cached
[0007/370] 2227_Elaphe_climacophora_std.json ... FAIL | 2 critical | 31880 in / 2495 out / 12032 cached
[0008/370] 2228_Elaphurus_davidianus_stx.json ... FAIL | 8 critical | 34743 in / 3645 out / 12032 cached
[0009/370] 2229_Elassodiscus_obscurus_abj.json ... FAIL | 12 critical | 34039 in / 3922 out / 12032 cached
[0010/370] 2230_Elassoma_zonatum_abj.json ... FAIL | 6 critical | 34535 in 

In [3]:
# ── Summary ───────────────────────────────────────────────────────────────────
from collections import Counter

counts = Counter()
for filepath in sorted(GENERATED_DIR.glob("*.json")):
    data = json.loads(filepath.read_text(encoding="utf-8"))
    if not data.get("generated_text"):
        counts["NO_GENERATED_TEXT"] += 1
        continue
    v = data.get("verification")
    if v is None:
        counts["NOT_VERIFIED"] += 1
    else:
        counts[v.get("verdict", "UNKNOWN")] += 1

total = sum(counts.values())
print(f"=== Verification Summary ({total} files) ===")
for verdict, count in counts.most_common():
    print(f"  {verdict}: {count}")

=== Verification Summary (370 files) ===
  FAIL: 335
  PASS: 35


In [4]:
# ── Detail: FAIL files ────────────────────────────────────────────────────────
fails = []
for filepath in sorted(GENERATED_DIR.glob("*.json")):
    data = json.loads(filepath.read_text(encoding="utf-8"))
    v = data.get("verification")
    if v and v.get("verdict") == "FAIL":
        fails.append({
            "file":            filepath.name,
            "specie":          data.get("specie"),
            "critical_errors": v.get("critical_errors"),
            "error_codes":     [e.get("code") for e in v.get("errors", [])],
        })

print(f"FAIL ({len(fails)} files):")
for r in fails:
    print(f"  {r['file']}  —  {r['specie']}  —  {r['critical_errors']} critical  —  {r['error_codes']}")

FAIL (335 files):
  2221_Elaenia_albiceps_std.json  —  Elaenia_albiceps  —  3 critical  —  ['A4', 'E2', 'E1', 'E2', 'H6']
  2222_Elaenia_gigas_std.json  —  Elaenia_gigas  —  5 critical  —  ['E2', 'G1c', 'G1e', 'D3', 'E6', 'E6', 'F5']
  2223_Elaenia_obscura_std.json  —  Elaenia_obscura  —  2 critical  —  ['A2', 'B3', 'C5', 'H6', 'H7', 'H8']
  2224_Elaenia_pallatangae_std.json  —  Elaenia_pallatangae  —  2 critical  —  ['G1f', 'H2', 'C1']
  2225_Elagatis_bipinnulata_abj.json  —  Elagatis_bipinnulata  —  3 critical  —  ['E4', 'E4', 'E6', 'F4']
  2226_Elanus_caeruleus_std.json  —  Elanus_caeruleus  —  2 critical  —  ['G1f', 'B3']
  2227_Elaphe_climacophora_std.json  —  Elaphe_climacophora  —  2 critical  —  ['C1', 'E2', 'G1f', 'H2', 'H3']
  2228_Elaphurus_davidianus_stx.json  —  Elaphurus_davidianus  —  8 critical  —  ['F5', 'F5', 'F5', 'F5', 'G2d', 'G2e', 'E2', 'E2', 'E2', 'E1', 'E1', 'E1', 'G1f', 'D3', 'D3', 'D3']
  2229_Elassodiscus_obscurus_abj.json  —  Elassodiscus_obscurus  —  12 cri

In [5]:
# ── Detail: GENERATION_FAILURE files ─────────────────────────────────────────
failures = []
for filepath in sorted(GENERATED_DIR.glob("*.json")):
    data = json.loads(filepath.read_text(encoding="utf-8"))
    v = data.get("verification")
    if v and v.get("verdict") == "GENERATION_FAILURE":
        failures.append({"file": filepath.name, "specie": data.get("specie")})

print(f"GENERATION_FAILURE ({len(failures)} files):")
for r in failures:
    print(f"  {r['file']}  —  {r['specie']}")

GENERATION_FAILURE (0 files):
